<a href="https://colab.research.google.com/github/SedexDevVE/SedexDev/blob/main/explorador_archivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from datetime import datetime

#Constantes que utilice

DIRECTORIO = '/content/drive/My Drive/InventarioColab'
EXTENSIONES_PERMITIDAS = ('.json', '.csv')
ARCHIVO_INVENTARIO = 'Datos.json'
FORMATO_FECHA = '%Y-%m-%d %H:%M:%S'

#Funcion de Google Drive

def montar_drive(punto_montaje='/content/drive'):

    try:
        from google.colab import drive  # verificar si solo existe en Google Colab
    except ImportError:
        print('Aviso: google.colab no disponible; se omite el montaje de Drive.')
        return False
    drive.mount(punto_montaje)
    return True


# Funciones o metodos auxiliares

def obtener_metadatos(ruta):
    tamano_kb = round(os.path.getsize(ruta) / 1024, 2)
    fecha = datetime.fromtimestamp(os.path.getmtime(ruta)).strftime(FORMATO_FECHA)
    return {'tamano_kb': tamano_kb, 'ultima_modificacion': fecha}


def validar_inventario(datos):

    if not isinstance(datos, list):
        return False, f'Se esperaba una lista y se obtuvo {type(datos).__name__}.'
    for i, registro in enumerate(datos):
        if not isinstance(registro, dict):
            return False, f'El registro {i} no es un objeto JSON (dict).'
    return True, f'Lista válida con {len(datos)} registro(s).'


def leer_inventario(ruta):
    resultado = {
        'encontrado': True,
        'valido': False,
        'total_registros': 0,
        'registros': [],
        'mensaje': '',
    }
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            datos = json.load(f)
    except json.JSONDecodeError as e:
        resultado['mensaje'] = f'JSON malformado: {e}'
        return resultado
    except OSError as e:
        resultado['mensaje'] = f'No se pudo leer el archivo: {e}'
        return resultado

    es_valido, mensaje = validar_inventario(datos)
    resultado['valido'] = es_valido
    resultado['mensaje'] = mensaje
    if es_valido:
        resultado['registros'] = datos
        resultado['total_registros'] = len(datos)
    return resultado


# Función principal

def list_files_in_directory(directorio=DIRECTORIO,
                            extensiones=EXTENSIONES_PERMITIDAS,
                            imprimir=True):

    extensiones = tuple(e.lower() for e in extensiones)
    resultado = {
        'directorio': directorio,
        'existe': False,
        'total_archivos': 0,
        'archivos': [],
        'inventario': {'encontrado': False},
        'error': None,
    }

    try:
        if str(directorio).startswith(('http://', 'https://')):
            raise FileNotFoundError
        nombres = sorted(os.listdir(directorio))
    except FileNotFoundError:
        resultado['error'] = ('Directorio no encontrado.')
    except NotADirectoryError:
        resultado['error'] = 'La ruta indicada no es un directorio.'
    except PermissionError:
        resultado['error'] = 'Sin permisos para leer el directorio.'
    else:
        resultado['existe'] = True
        for nombre in nombres:
            ruta = os.path.join(directorio, nombre)
            extension = os.path.splitext(nombre)[1].lower()
            if not os.path.isfile(ruta) or extension not in extensiones:
                continue  # filtrado selectivo
            info = {'nombre': nombre, 'extension': extension}
            info.update(obtener_metadatos(ruta))
            resultado['archivos'].append(info)
        resultado['total_archivos'] = len(resultado['archivos'])

        ruta_inv = os.path.join(directorio, ARCHIVO_INVENTARIO)
        if os.path.isfile(ruta_inv):
            resultado['inventario'] = leer_inventario(ruta_inv)

    if imprimir:
        _imprimir_resultado(resultado)
    return resultado


def _imprimir_resultado(r):

    if r['error']:
        print(r['error'])
        return
    print(f"Archivos {', '.join(EXTENSIONES_PERMITIDAS)} en {r['directorio']}:")
    for a in r['archivos']:
        print(f" └── {a['nombre']}  |  {a['tamano_kb']} KB  |  "
              f"modificado: {a['ultima_modificacion']}")
    print(f"Total: {r['total_archivos']} archivo(s).")
    inv = r['inventario']
    if inv['encontrado']:
        estado = 'VÁLIDO' if inv['valido'] else 'INVÁLIDO'
        print(f"\n{ARCHIVO_INVENTARIO}: {estado} - {inv['mensaje']}")
    else:
        print(f'\n{ARCHIVO_INVENTARIO} no encontrado en la carpeta.')


if __name__ == '__main__':
    montar_drive()
    datos = list_files_in_directory(DIRECTORIO)
    # Salida lista para una API (JSON)
    print('\nRespuesta estructurada:')
    print(json.dumps(datos, indent=2, ensure_ascii=False))